In [1]:
# ============================================================
# DEMAND V3 — FOMO + TTFL + TRAFFIC + CONTACT RATE
# FULL PIPELINE FROM SCRATCH
#
# Goal:
# - Check "demand" more carefully for supply-demand analysis
# - Combine:
#   1. Supply: listing_count
#   2. FOMO: early contact rate in first N days
#   3. TTFL: time to first lead
#   4. Traffic: pageviews
#   5. CR: strict_contact / pageviews
#
# Important:
# - Filter by listing_count threshold to avoid tiny groups creating fake insights
# - Uses strict_contact only:
#   view_phone + contact_chat + contact_zalo + contact_sms
# - Keeps other_interaction only as optional all_positive metric
# ============================================================

# ============================================================
# 0. IMPORTS
# ============================================================

import os
import gc
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe"

# ============================================================
# 1. CONFIG — EDIT HERE IF NEEDED
# ============================================================

DIM_LISTING_PATH = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/dim_listing"
SNAPSHOT_PATH    = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/fact_listing_snapshot"
EVENTS_PATH      = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events"

CACHE_DIR = "/kaggle/working/demand_v3_fomo_ttfl_traffic"

FORCE_REBUILD = True

# Batch size: giảm nếu bị OOM
DIM_BATCH_SIZE      = 500_000
SNAPSHOT_BATCH_SIZE = 500_000
EVENTS_BATCH_SIZE   = 300_000

# FOMO window
FOMO_MAX_AGE_DAYS = 7

# Grouping option:
# True  = city/category/ad_type, district/category/ad_type
# False = city/category, district/category
GROUP_WITH_AD_TYPE = True

# Listing threshold để tránh group quá nhỏ
MIN_LISTINGS_CITY_GROUP = 1000
MIN_LISTINGS_HCM_GROUP  = 500

# Evidence thresholds.
# Có thể set về 0 nếu chỉ muốn dùng listing_count threshold.
MIN_PAGEVIEWS_CITY_GROUP        = 1000
MIN_EARLY_VIEWS_CITY_GROUP     = 500
MIN_ITEMS_WITH_LEAD_CITY_GROUP = 30

MIN_PAGEVIEWS_HCM_GROUP        = 500
MIN_EARLY_VIEWS_HCM_GROUP     = 300
MIN_ITEMS_WITH_LEAD_HCM_GROUP = 20

STRICT_CONTACTS = {
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
}

ALL_POSITIVE = STRICT_CONTACTS.union({"other_interaction"})

CATEGORY_MAP = {
    1010: "1010 - Phòng trọ / căn hộ thuê",
    1020: "1020 - Căn hộ / chung cư",
    1030: "1030 - Nhà ở",
    1040: "1040 - Đất / BĐS thương mại",
    1050: "1050 - Dự án / căn hộ mới",
}

# ============================================================
# 2. RESET CACHE
# ============================================================

if FORCE_REBUILD and os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)

os.makedirs(CACHE_DIR, exist_ok=True)

# ============================================================
# 3. HELPERS
# ============================================================

def clean_text(x, unknown="Unknown"):
    if pd.isna(x):
        return unknown
    x = str(x).strip()
    if x == "":
        return unknown
    return x


def clean_city(x):
    if pd.isna(x):
        return "Unknown"
    x = str(x).strip()
    if x == "":
        return "Unknown"

    xl = x.lower()

    if (
        "hồ chí minh" in xl
        or "ho chi minh" in xl
        or "tp.hcm" in xl
        or "tp hcm" in xl
        or xl == "hcm"
        or xl == "tphcm"
    ):
        return "Tp Hồ Chí Minh"

    return x


def clean_district(x):
    return clean_text(x, "Unknown District")


def clean_category(x):
    if pd.isna(x):
        return "Unknown Category"
    try:
        x = int(float(x))
        return CATEGORY_MAP.get(x, str(x))
    except Exception:
        return "Unknown Category"


def safe_div(num, den):
    return num / den.replace(0, np.nan)


def acc_add(acc, key, row, metric_cols):
    if key not in acc:
        acc[key] = {c: 0.0 for c in metric_cols}

    for c in metric_cols:
        acc[key][c] += float(row[c])


def acc_to_df(acc, key_cols):
    rows = []
    for key, values in acc.items():
        row = dict(zip(key_cols, key))
        row.update(values)
        rows.append(row)
    return pd.DataFrame(rows)


def add_group_label(df, geo_col):
    df = df.copy()

    parts = [df[geo_col].astype(str), df["category_name"].astype(str)]

    if GROUP_WITH_AD_TYPE and "ad_type" in df.columns:
        parts.append(df["ad_type"].astype(str))

    label = parts[0]
    for p in parts[1:]:
        label = label + " | " + p

    df["group_label"] = label
    return df


def pct_rank(series, higher_is_better=True):
    s = series.replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() <= 1:
        return pd.Series(np.nan, index=s.index)

    if higher_is_better:
        return s.rank(pct=True, ascending=True)
    else:
        # lower value should get higher score
        return s.rank(pct=True, ascending=False)


def build_score_and_segment(
    df,
    geo_col,
    min_listings,
    min_pageviews,
    min_early_views,
    min_items_with_lead,
):
    df = df.copy()

    # -----------------------------
    # Basic ratios
    # -----------------------------
    df["pageviews_per_listing"] = safe_div(
        df["pageviews"],
        df["listing_count"]
    )

    df["strict_contact_per_listing"] = safe_div(
        df["strict_contact"],
        df["listing_count"]
    )

    df["all_positive_per_listing"] = safe_div(
        df["all_positive"],
        df["listing_count"]
    )

    df["strict_contact_rate"] = safe_div(
        df["strict_contact"],
        df["pageviews"]
    )

    df["all_positive_rate"] = safe_div(
        df["all_positive"],
        df["pageviews"]
    )

    df["fomo_early_contact_rate"] = safe_div(
        df["early_contacts_24h"],
        df["early_views_24h"]
    )

    df["lead_listing_rate"] = safe_div(
        df["items_with_lead"],
        df["listing_count"]
    )

    # -----------------------------
    # Reliability filters
    # -----------------------------
    df["passes_listing_threshold"] = (
        df["listing_count"].fillna(0) >= min_listings
    )

    df["passes_evidence_threshold"] = (
        (df["pageviews"].fillna(0) >= min_pageviews)
        & (df["early_views_24h"].fillna(0) >= min_early_views)
        & (df["items_with_lead"].fillna(0) >= min_items_with_lead)
    )

    df["analysis_eligible"] = (
        df["passes_listing_threshold"]
        & df["passes_evidence_threshold"]
        & df["fomo_early_contact_rate"].notna()
        & df["strict_contact_rate"].notna()
        & df["strict_contact_per_listing"].notna()
        & df["median_ttfl_days"].notna()
    )

    # -----------------------------
    # Rank-based demand score
    # -----------------------------
    score_cols = [
        "score_fomo",
        "score_strict_cr",
        "score_contact_per_listing",
        "score_ttfl_fast",
        "score_pageviews_per_listing",
    ]

    for c in score_cols:
        df[c] = np.nan

    idx = df["analysis_eligible"]

    df.loc[idx, "score_fomo"] = pct_rank(
        df.loc[idx, "fomo_early_contact_rate"],
        higher_is_better=True
    )

    df.loc[idx, "score_strict_cr"] = pct_rank(
        df.loc[idx, "strict_contact_rate"],
        higher_is_better=True
    )

    df.loc[idx, "score_contact_per_listing"] = pct_rank(
        df.loc[idx, "strict_contact_per_listing"],
        higher_is_better=True
    )

    df.loc[idx, "score_ttfl_fast"] = pct_rank(
        df.loc[idx, "median_ttfl_days"],
        higher_is_better=False
    )

    df.loc[idx, "score_pageviews_per_listing"] = pct_rank(
        df.loc[idx, "pageviews_per_listing"],
        higher_is_better=True
    )

    weights = {
        "score_fomo": 0.25,
        "score_strict_cr": 0.25,
        "score_contact_per_listing": 0.25,
        "score_ttfl_fast": 0.20,
        "score_pageviews_per_listing": 0.05,
    }

    weighted_sum = 0
    weight_used = 0

    for c, w in weights.items():
        valid_c = df[c].notna()
        weighted_sum = weighted_sum + df[c].fillna(0) * w
        weight_used = weight_used + valid_c.astype(float) * w

    df["demand_score"] = weighted_sum / weight_used.replace(0, np.nan)

    # -----------------------------
    # Segment labels
    # -----------------------------
    valid = df[df["analysis_eligible"]].copy()

    if len(valid) == 0:
        df["demand_segment"] = np.where(
            df["passes_listing_threshold"],
            "Enough listings but weak evidence",
            "Too small / unreliable"
        )
        return add_group_label(df, geo_col)

    q = {
        "fomo_hi": valid["fomo_early_contact_rate"].quantile(0.60),
        "fomo_lo": valid["fomo_early_contact_rate"].quantile(0.40),
        "cr_hi": valid["strict_contact_rate"].quantile(0.60),
        "cr_lo": valid["strict_contact_rate"].quantile(0.40),
        "cpl_hi": valid["strict_contact_per_listing"].quantile(0.60),
        "cpl_lo": valid["strict_contact_per_listing"].quantile(0.40),
        "ttfl_fast": valid["median_ttfl_days"].quantile(0.40),
        "ttfl_slow": valid["median_ttfl_days"].quantile(0.60),
        "traffic_hi": valid["pageviews_per_listing"].quantile(0.60),
        "traffic_lo": valid["pageviews_per_listing"].quantile(0.40),
        "supply_hi": valid["listing_count"].quantile(0.60),
    }

    def label(row):
        if not row["passes_listing_threshold"]:
            return "Too small / unreliable"

        if not row["passes_evidence_threshold"]:
            return "Enough listings but weak evidence"

        if not row["analysis_eligible"]:
            return "Missing key metric"

        high_fomo = row["fomo_early_contact_rate"] >= q["fomo_hi"]
        high_cr = row["strict_contact_rate"] >= q["cr_hi"]
        high_cpl = row["strict_contact_per_listing"] >= q["cpl_hi"]
        fast_ttfl = row["median_ttfl_days"] <= q["ttfl_fast"]

        low_cr = row["strict_contact_rate"] <= q["cr_lo"]
        low_cpl = row["strict_contact_per_listing"] <= q["cpl_lo"]
        slow_ttfl = row["median_ttfl_days"] >= q["ttfl_slow"]

        high_traffic = row["pageviews_per_listing"] >= q["traffic_hi"]
        low_traffic = row["pageviews_per_listing"] <= q["traffic_lo"]
        high_supply = row["listing_count"] >= q["supply_hi"]

        if high_fomo and high_cr and high_cpl and fast_ttfl:
            return "Confirmed hot demand"

        if high_traffic and low_cr:
            return "High attention, low conversion"

        if high_fomo and fast_ttfl and low_traffic:
            return "FOMO fast but traffic thin"

        if high_supply and low_cpl and slow_ttfl:
            return "Potential oversupply / slow liquidity"

        if high_cpl and fast_ttfl:
            return "Fast liquidity"

        if low_cpl and slow_ttfl:
            return "Weak demand / slow liquidity"

        return "Normal / mixed"

    df["demand_segment"] = df.apply(label, axis=1)

    return add_group_label(df, geo_col)


def save_plot(fig, name):
    path = os.path.join(CACHE_DIR, name)
    fig.write_html(path)
    fig.show(renderer="iframe")
    print("[SAVED]", path)


# ============================================================
# 4. FIND INPUT FILES
# ============================================================

print("Available Kaggle input folders:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    if filenames:
        print(dirname, "| sample:", filenames[:2])

# ============================================================
# 5. BUILD item_map FROM dim_listing
# ============================================================

print("\n[1/7] Building item_map from dim_listing...")

dim_ds = ds.dataset(DIM_LISTING_PATH, format="parquet")

dim_cols = [
    "item_id",
    "city_name",
    "district_name",
    "category",
    "ad_type",
    "posted_date",
]

item_parts = []

for i, batch in enumerate(
    dim_ds.to_batches(columns=dim_cols, batch_size=DIM_BATCH_SIZE),
    start=1
):
    df = batch.to_pandas()

    df["city_name"] = df["city_name"].map(clean_city)
    df["district_name"] = df["district_name"].map(clean_district)
    df["category_name"] = df["category"].map(clean_category)
    df["ad_type"] = df["ad_type"].map(lambda x: clean_text(x, "Unknown Ad Type"))
    df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")

    df = df[
        [
            "item_id",
            "city_name",
            "district_name",
            "category",
            "category_name",
            "ad_type",
            "posted_date",
        ]
    ].drop_duplicates("item_id")

    item_parts.append(df)

    if i % 10 == 0:
        print("dim_listing batches:", i)

    del df
    gc.collect()

item_map = pd.concat(item_parts, ignore_index=True).drop_duplicates("item_id")

item_map_path = os.path.join(CACHE_DIR, "item_map.parquet")
item_map.to_parquet(item_map_path, index=False)

print("item_map shape:", item_map.shape)
display(item_map.head())

# ============================================================
# 6. GROUP KEYS
# ============================================================

city_keys = ["city_name", "category_name"]
hcm_keys = ["district_name", "category_name"]

if GROUP_WITH_AD_TYPE:
    city_keys.append("ad_type")
    hcm_keys.append("ad_type")

print("City group keys:", city_keys)
print("HCM group keys:", hcm_keys)

# ============================================================
# 7. BUILD SUPPLY TABLES FROM dim_listing
# ============================================================

print("\n[2/7] Building supply tables...")

city_supply = (
    item_map
    .groupby(city_keys, dropna=False)
    .agg(listing_count=("item_id", "nunique"))
    .reset_index()
)

hcm_supply = (
    item_map[
        (item_map["city_name"] == "Tp Hồ Chí Minh")
        & (~item_map["district_name"].isin(["Unknown District", "Unmapped Item"]))
    ]
    .groupby(hcm_keys, dropna=False)
    .agg(listing_count=("item_id", "nunique"))
    .reset_index()
)

city_supply.to_parquet(os.path.join(CACHE_DIR, "city_supply.parquet"), index=False)
hcm_supply.to_parquet(os.path.join(CACHE_DIR, "hcm_supply.parquet"), index=False)

print("city_supply:", city_supply.shape)
print("hcm_supply:", hcm_supply.shape)

display(city_supply.sort_values("listing_count", ascending=False).head(20))
display(hcm_supply.sort_values("listing_count", ascending=False).head(20))

# ============================================================
# 8. SCAN fact_listing_snapshot ONCE
#    Build FOMO + item-level TTFL
# ============================================================

print("\n[3/7] Scanning fact_listing_snapshot for FOMO + TTFL...")

snap_ds = ds.dataset(SNAPSHOT_PATH, format="parquet")

item_map_for_snapshot = item_map[
    [
        "item_id",
        "city_name",
        "district_name",
        "category_name",
        "ad_type",
        "posted_date",
    ]
].copy()

city_fomo_acc = {}
hcm_fomo_acc = {}

fomo_metric_cols = [
    "early_views_24h",
    "early_contacts_24h",
    "early_snapshot_rows",
    "early_unique_items_sum_batch",
    "missing_views_rows",
    "missing_contacts_rows",
]

first_lead = {}

dq = {
    "total_snapshot_rows": 0,
    "unmapped_item_rows": 0,
    "negative_age_rows": 0,
    "missing_views_rows": 0,
    "missing_contacts_rows": 0,
    "valid_fomo_rows": 0,
    "valid_first_lead_rows": 0,
}

snapshot_cols = [
    "item_id",
    "date",
    "views_24h",
    "contacts_24h",
    "listing_age_days",
]

for i, batch in enumerate(
    snap_ds.to_batches(columns=snapshot_cols, batch_size=SNAPSHOT_BATCH_SIZE),
    start=1
):
    df = batch.to_pandas()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["views_24h"] = pd.to_numeric(df["views_24h"], errors="coerce")
    df["contacts_24h"] = pd.to_numeric(df["contacts_24h"], errors="coerce")
    df["listing_age_days"] = pd.to_numeric(df["listing_age_days"], errors="coerce")

    dq["total_snapshot_rows"] += len(df)
    dq["negative_age_rows"] += int((df["listing_age_days"] < 0).sum())
    dq["missing_views_rows"] += int(df["views_24h"].isna().sum())
    dq["missing_contacts_rows"] += int(df["contacts_24h"].isna().sum())

    df = df.merge(
        item_map_for_snapshot,
        on="item_id",
        how="left",
        indicator=True
    )

    df["is_mapped_item"] = df["_merge"].eq("both")
    dq["unmapped_item_rows"] += int((~df["is_mapped_item"]).sum())

    # -----------------------------
    # 8A. FOMO rows
    # Keep NaN views/contact as 0.
    # Only remove negative / invalid age.
    # -----------------------------
    valid_fomo = df[
        (df["is_mapped_item"])
        & (df["listing_age_days"].notna())
        & (df["listing_age_days"] >= 0)
        & (df["listing_age_days"] <= FOMO_MAX_AGE_DAYS)
    ].copy()

    dq["valid_fomo_rows"] += len(valid_fomo)

    if len(valid_fomo) > 0:
        valid_fomo["views_24h_clean"] = valid_fomo["views_24h"].fillna(0)
        valid_fomo["contacts_24h_clean"] = valid_fomo["contacts_24h"].fillna(0)

        # City-level FOMO
        grp = (
            valid_fomo
            .groupby(city_keys, dropna=False)
            .agg(
                early_views_24h=("views_24h_clean", "sum"),
                early_contacts_24h=("contacts_24h_clean", "sum"),
                early_snapshot_rows=("item_id", "size"),
                early_unique_items_sum_batch=("item_id", "nunique"),
                missing_views_rows=("views_24h", lambda x: x.isna().sum()),
                missing_contacts_rows=("contacts_24h", lambda x: x.isna().sum()),
            )
            .reset_index()
        )

        for _, row in grp.iterrows():
            key = tuple(row[k] for k in city_keys)
            acc_add(city_fomo_acc, key, row, fomo_metric_cols)

        # HCM district-level FOMO
        hcm_valid = valid_fomo[
            (valid_fomo["city_name"] == "Tp Hồ Chí Minh")
            & (~valid_fomo["district_name"].isin(["Unknown District", "Unmapped Item"]))
        ].copy()

        if len(hcm_valid) > 0:
            grp = (
                hcm_valid
                .groupby(hcm_keys, dropna=False)
                .agg(
                    early_views_24h=("views_24h_clean", "sum"),
                    early_contacts_24h=("contacts_24h_clean", "sum"),
                    early_snapshot_rows=("item_id", "size"),
                    early_unique_items_sum_batch=("item_id", "nunique"),
                    missing_views_rows=("views_24h", lambda x: x.isna().sum()),
                    missing_contacts_rows=("contacts_24h", lambda x: x.isna().sum()),
                )
                .reset_index()
            )

            for _, row in grp.iterrows():
                key = tuple(row[k] for k in hcm_keys)
                acc_add(hcm_fomo_acc, key, row, fomo_metric_cols)

    # -----------------------------
    # 8B. TTFL item-level
    # First lead = minimum listing_age_days where contacts_24h > 0
    # -----------------------------
    lead_rows = df[
        (df["is_mapped_item"])
        & (df["listing_age_days"].notna())
        & (df["listing_age_days"] >= 0)
        & (df["contacts_24h"].notna())
        & (df["contacts_24h"] > 0)
    ][["item_id", "listing_age_days"]].copy()

    dq["valid_first_lead_rows"] += len(lead_rows)

    if len(lead_rows) > 0:
        first_in_batch = lead_rows.groupby("item_id")["listing_age_days"].min()

        for item_id, age in first_in_batch.items():
            old = first_lead.get(item_id)
            if old is None or age < old:
                first_lead[item_id] = float(age)

    if i % 20 == 0:
        print("snapshot batches:", i)

    del df, valid_fomo
    gc.collect()

city_fomo = acc_to_df(city_fomo_acc, city_keys)
hcm_fomo = acc_to_df(hcm_fomo_acc, hcm_keys)

item_first_lead = pd.DataFrame({
    "item_id": list(first_lead.keys()),
    "time_to_first_lead_days": list(first_lead.values()),
})

city_fomo.to_parquet(os.path.join(CACHE_DIR, "city_fomo.parquet"), index=False)
hcm_fomo.to_parquet(os.path.join(CACHE_DIR, "hcm_fomo.parquet"), index=False)
item_first_lead.to_parquet(os.path.join(CACHE_DIR, "item_first_lead.parquet"), index=False)

dq_df = pd.DataFrame([dq])
dq_df["valid_fomo_rate"] = dq_df["valid_fomo_rows"] / dq_df["total_snapshot_rows"]
dq_df["valid_first_lead_rate"] = dq_df["valid_first_lead_rows"] / dq_df["total_snapshot_rows"]
dq_df.to_csv(os.path.join(CACHE_DIR, "snapshot_dq_summary.csv"), index=False)

print("city_fomo:", city_fomo.shape)
print("hcm_fomo:", hcm_fomo.shape)
print("item_first_lead:", item_first_lead.shape)
display(dq_df)

# ============================================================
# 9. BUILD TTFL GROUP TABLES
# ============================================================

print("\n[4/7] Building TTFL group tables...")

ttfl = item_first_lead.merge(
    item_map[
        [
            "item_id",
            "city_name",
            "district_name",
            "category_name",
            "ad_type",
        ]
    ],
    on="item_id",
    how="left"
)

ttfl["time_to_first_lead_days"] = pd.to_numeric(
    ttfl["time_to_first_lead_days"],
    errors="coerce"
)

ttfl = ttfl[
    (ttfl["time_to_first_lead_days"].notna())
    & (ttfl["time_to_first_lead_days"] >= 0)
].copy()

city_ttfl = (
    ttfl
    .groupby(city_keys, dropna=False)
    .agg(
        items_with_lead=("item_id", "nunique"),
        median_ttfl_days=("time_to_first_lead_days", "median"),
        mean_ttfl_days=("time_to_first_lead_days", "mean"),
        p25_ttfl_days=("time_to_first_lead_days", lambda x: x.quantile(0.25)),
        p75_ttfl_days=("time_to_first_lead_days", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

hcm_ttfl = (
    ttfl[
        (ttfl["city_name"] == "Tp Hồ Chí Minh")
        & (~ttfl["district_name"].isin(["Unknown District", "Unmapped Item"]))
    ]
    .groupby(hcm_keys, dropna=False)
    .agg(
        items_with_lead=("item_id", "nunique"),
        median_ttfl_days=("time_to_first_lead_days", "median"),
        mean_ttfl_days=("time_to_first_lead_days", "mean"),
        p25_ttfl_days=("time_to_first_lead_days", lambda x: x.quantile(0.25)),
        p75_ttfl_days=("time_to_first_lead_days", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

city_ttfl.to_parquet(os.path.join(CACHE_DIR, "city_ttfl.parquet"), index=False)
hcm_ttfl.to_parquet(os.path.join(CACHE_DIR, "hcm_ttfl.parquet"), index=False)

print("city_ttfl:", city_ttfl.shape)
print("hcm_ttfl:", hcm_ttfl.shape)

display(city_ttfl.sort_values("items_with_lead", ascending=False).head(20))
display(hcm_ttfl.sort_values("items_with_lead", ascending=False).head(20))

# ============================================================
# 10. SCAN fact_user_events FOR TRAFFIC + CONTACT RATE
# ============================================================

print("\n[5/7] Scanning fact_user_events for traffic + strict contact rate...")

events_ds = ds.dataset(EVENTS_PATH, format="parquet")

item_map_for_events = item_map[
    [
        "item_id",
        "city_name",
        "district_name",
        "category_name",
        "ad_type",
    ]
].copy()

traffic_metric_cols = [
    "pageviews",
    "strict_contact",
    "all_positive",
    "total_events",
]

city_traffic_acc = {}
hcm_traffic_acc = {}

event_cols = [
    "item_id",
    "city_name",
    "category",
    "event_type",
]

for i, batch in enumerate(
    events_ds.to_batches(columns=event_cols, batch_size=EVENTS_BATCH_SIZE),
    start=1
):
    df = batch.to_pandas()

    df = df.rename(columns={
        "city_name": "event_city_name",
        "category": "event_category",
    })

    df["event_city_name"] = df["event_city_name"].map(clean_city)
    df["event_category_name"] = df["event_category"].map(clean_category)
    df["event_type"] = df["event_type"].fillna("unknown").astype(str)

    df = df.merge(
        item_map_for_events,
        on="item_id",
        how="left"
    )

    # Prefer dim_listing metadata.
    # If item is unmapped, fallback to event city/category where available.
    df["city_name"] = df["city_name"].fillna(df["event_city_name"])
    df["category_name"] = df["category_name"].fillna(df["event_category_name"])
    df["district_name"] = df["district_name"].fillna("Unknown District")
    df["ad_type"] = df["ad_type"].fillna("Unknown Ad Type")

    df["pageviews"] = (df["event_type"] == "pageview").astype(np.int64)
    df["strict_contact"] = df["event_type"].isin(STRICT_CONTACTS).astype(np.int64)
    df["all_positive"] = df["event_type"].isin(ALL_POSITIVE).astype(np.int64)
    df["total_events"] = 1

    # City traffic
    grp = (
        df
        .groupby(city_keys, dropna=False)[traffic_metric_cols]
        .sum()
        .reset_index()
    )

    for _, row in grp.iterrows():
        key = tuple(row[k] for k in city_keys)
        acc_add(city_traffic_acc, key, row, traffic_metric_cols)

    # HCM district traffic
    hcm_df = df[
        (df["city_name"] == "Tp Hồ Chí Minh")
        & (~df["district_name"].isin(["Unknown District", "Unmapped Item"]))
    ].copy()

    if len(hcm_df) > 0:
        grp = (
            hcm_df
            .groupby(hcm_keys, dropna=False)[traffic_metric_cols]
            .sum()
            .reset_index()
        )

        for _, row in grp.iterrows():
            key = tuple(row[k] for k in hcm_keys)
            acc_add(hcm_traffic_acc, key, row, traffic_metric_cols)

    if i % 20 == 0:
        print("event batches:", i)

    del df
    gc.collect()

city_traffic = acc_to_df(city_traffic_acc, city_keys)
hcm_traffic = acc_to_df(hcm_traffic_acc, hcm_keys)

city_traffic.to_parquet(os.path.join(CACHE_DIR, "city_traffic.parquet"), index=False)
hcm_traffic.to_parquet(os.path.join(CACHE_DIR, "hcm_traffic.parquet"), index=False)

print("city_traffic:", city_traffic.shape)
print("hcm_traffic:", hcm_traffic.shape)

display(city_traffic.sort_values("pageviews", ascending=False).head(20))
display(hcm_traffic.sort_values("pageviews", ascending=False).head(20))

# ============================================================
# 11. MERGE DEMAND V3 TABLES
# ============================================================

print("\n[6/7] Merging supply + FOMO + TTFL + traffic...")

count_cols = [
    "listing_count",
    "early_views_24h",
    "early_contacts_24h",
    "early_snapshot_rows",
    "early_unique_items_sum_batch",
    "missing_views_rows",
    "missing_contacts_rows",
    "items_with_lead",
    "pageviews",
    "strict_contact",
    "all_positive",
    "total_events",
]

city_demand = (
    city_supply
    .merge(city_fomo, on=city_keys, how="outer")
    .merge(city_ttfl, on=city_keys, how="outer")
    .merge(city_traffic, on=city_keys, how="outer")
)

hcm_demand = (
    hcm_supply
    .merge(hcm_fomo, on=hcm_keys, how="outer")
    .merge(hcm_ttfl, on=hcm_keys, how="outer")
    .merge(hcm_traffic, on=hcm_keys, how="outer")
)

for df in [city_demand, hcm_demand]:
    for c in count_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

city_demand = build_score_and_segment(
    city_demand,
    geo_col="city_name",
    min_listings=MIN_LISTINGS_CITY_GROUP,
    min_pageviews=MIN_PAGEVIEWS_CITY_GROUP,
    min_early_views=MIN_EARLY_VIEWS_CITY_GROUP,
    min_items_with_lead=MIN_ITEMS_WITH_LEAD_CITY_GROUP,
)

hcm_demand = build_score_and_segment(
    hcm_demand,
    geo_col="district_name",
    min_listings=MIN_LISTINGS_HCM_GROUP,
    min_pageviews=MIN_PAGEVIEWS_HCM_GROUP,
    min_early_views=MIN_EARLY_VIEWS_HCM_GROUP,
    min_items_with_lead=MIN_ITEMS_WITH_LEAD_HCM_GROUP,
)

city_result_path = os.path.join(CACHE_DIR, "city_demand_v3_fomo_ttfl_traffic.csv")
hcm_result_path = os.path.join(CACHE_DIR, "hcm_district_demand_v3_fomo_ttfl_traffic.csv")

city_demand.to_csv(city_result_path, index=False)
hcm_demand.to_csv(hcm_result_path, index=False)

print("[SAVED]", city_result_path)
print("[SAVED]", hcm_result_path)

# ============================================================
# 12. MAIN TABLES TO INSPECT
# ============================================================

print("\nTop confirmed / high-score city groups:")
top_city = (
    city_demand[
        city_demand["analysis_eligible"]
    ]
    .sort_values("demand_score", ascending=False)
    .head(30)
)

display(
    top_city[
        [
            "group_label",
            "listing_count",
            "pageviews",
            "strict_contact",
            "strict_contact_rate",
            "strict_contact_per_listing",
            "early_views_24h",
            "early_contacts_24h",
            "fomo_early_contact_rate",
            "items_with_lead",
            "lead_listing_rate",
            "median_ttfl_days",
            "demand_score",
            "demand_segment",
        ]
    ]
)

print("\nTop confirmed / high-score HCM district groups:")
top_hcm = (
    hcm_demand[
        hcm_demand["analysis_eligible"]
    ]
    .sort_values("demand_score", ascending=False)
    .head(30)
)

display(
    top_hcm[
        [
            "group_label",
            "listing_count",
            "pageviews",
            "strict_contact",
            "strict_contact_rate",
            "strict_contact_per_listing",
            "early_views_24h",
            "early_contacts_24h",
            "fomo_early_contact_rate",
            "items_with_lead",
            "lead_listing_rate",
            "median_ttfl_days",
            "demand_score",
            "demand_segment",
        ]
    ]
)

print("\nPotential high attention but low conversion — city:")
display(
    city_demand[
        city_demand["analysis_eligible"]
        & (city_demand["demand_segment"] == "High attention, low conversion")
    ]
    .sort_values("pageviews_per_listing", ascending=False)
    .head(20)
    [
        [
            "group_label",
            "listing_count",
            "pageviews_per_listing",
            "strict_contact_rate",
            "strict_contact_per_listing",
            "median_ttfl_days",
            "demand_segment",
        ]
    ]
)

print("\nPotential oversupply / slow liquidity — HCM:")
display(
    hcm_demand[
        hcm_demand["analysis_eligible"]
        & (hcm_demand["demand_segment"] == "Potential oversupply / slow liquidity")
    ]
    .sort_values(["listing_count", "strict_contact_per_listing"], ascending=[False, True])
    .head(20)
    [
        [
            "group_label",
            "listing_count",
            "pageviews",
            "strict_contact_rate",
            "strict_contact_per_listing",
            "median_ttfl_days",
            "demand_segment",
        ]
    ]
)

# Save key filtered tables
top_city.to_csv(os.path.join(CACHE_DIR, "top_city_confirmed_demand.csv"), index=False)
top_hcm.to_csv(os.path.join(CACHE_DIR, "top_hcm_confirmed_demand.csv"), index=False)

city_demand[
    ~city_demand["passes_listing_threshold"]
].to_csv(os.path.join(CACHE_DIR, "city_groups_too_small.csv"), index=False)

hcm_demand[
    ~hcm_demand["passes_listing_threshold"]
].to_csv(os.path.join(CACHE_DIR, "hcm_groups_too_small.csv"), index=False)

# ============================================================
# 13. PLOTS
# ============================================================

print("\n[7/7] Drawing charts...")

# -----------------------------
# 13A. City scatter: supply vs demand score
# -----------------------------
plot_city = city_demand[
    city_demand["passes_listing_threshold"]
].copy()

fig = px.scatter(
    plot_city,
    x="listing_count",
    y="demand_score",
    size="pageviews",
    color="demand_segment",
    hover_name="group_label",
    hover_data={
        "listing_count": ":,.0f",
        "pageviews": ":,.0f",
        "strict_contact": ":,.0f",
        "strict_contact_rate": ":.4f",
        "strict_contact_per_listing": ":.4f",
        "fomo_early_contact_rate": ":.4f",
        "median_ttfl_days": ":.2f",
        "items_with_lead": ":,.0f",
        "demand_score": ":.3f",
    },
    log_x=True,
    title="City Group Demand V3: Supply vs Combined Demand Score",
)

fig.update_layout(height=650)
save_plot(fig, "city_supply_vs_demand_score.html")

# -----------------------------
# 13B. HCM scatter: supply vs demand score
# -----------------------------
plot_hcm = hcm_demand[
    hcm_demand["passes_listing_threshold"]
].copy()

fig = px.scatter(
    plot_hcm,
    x="listing_count",
    y="demand_score",
    size="pageviews",
    color="demand_segment",
    hover_name="group_label",
    hover_data={
        "listing_count": ":,.0f",
        "pageviews": ":,.0f",
        "strict_contact": ":,.0f",
        "strict_contact_rate": ":.4f",
        "strict_contact_per_listing": ":.4f",
        "fomo_early_contact_rate": ":.4f",
        "median_ttfl_days": ":.2f",
        "items_with_lead": ":,.0f",
        "demand_score": ":.3f",
    },
    log_x=True,
    title="HCM District Group Demand V3: Supply vs Combined Demand Score",
)

fig.update_layout(height=650)
save_plot(fig, "hcm_supply_vs_demand_score.html")

# -----------------------------
# 13C. Top City groups by demand score
# -----------------------------
plot_top = (
    city_demand[
        city_demand["analysis_eligible"]
    ]
    .sort_values("demand_score", ascending=False)
    .head(25)
    .sort_values("demand_score", ascending=True)
)

fig = px.bar(
    plot_top,
    x="demand_score",
    y="group_label",
    orientation="h",
    color="demand_segment",
    hover_data={
        "listing_count": ":,.0f",
        "pageviews": ":,.0f",
        "strict_contact_rate": ":.4f",
        "fomo_early_contact_rate": ":.4f",
        "median_ttfl_days": ":.2f",
    },
    title="Top City Groups by Demand V3 Score",
)

fig.update_layout(height=750, yaxis_title="")
save_plot(fig, "top_city_demand_score.html")

# -----------------------------
# 13D. Top HCM groups by demand score
# -----------------------------
plot_top = (
    hcm_demand[
        hcm_demand["analysis_eligible"]
    ]
    .sort_values("demand_score", ascending=False)
    .head(25)
    .sort_values("demand_score", ascending=True)
)

fig = px.bar(
    plot_top,
    x="demand_score",
    y="group_label",
    orientation="h",
    color="demand_segment",
    hover_data={
        "listing_count": ":,.0f",
        "pageviews": ":,.0f",
        "strict_contact_rate": ":.4f",
        "fomo_early_contact_rate": ":.4f",
        "median_ttfl_days": ":.2f",
    },
    title="Top HCM District Groups by Demand V3 Score",
)

fig.update_layout(height=750, yaxis_title="")
save_plot(fig, "top_hcm_demand_score.html")

# -----------------------------
# 13E. FOMO vs TTFL map — HCM
# x high = stronger FOMO
# y low = faster lead
# -----------------------------
plot_hcm2 = hcm_demand[
    hcm_demand["analysis_eligible"]
].copy()

fig = px.scatter(
    plot_hcm2,
    x="fomo_early_contact_rate",
    y="median_ttfl_days",
    size="listing_count",
    color="demand_segment",
    hover_name="group_label",
    hover_data={
        "listing_count": ":,.0f",
        "pageviews": ":,.0f",
        "strict_contact_rate": ":.4f",
        "strict_contact_per_listing": ":.4f",
        "items_with_lead": ":,.0f",
        "demand_score": ":.3f",
    },
    title="HCM Demand Map: FOMO Rate vs Median TTFL<br>Right = stronger early demand, Lower = faster lead",
)

fig.update_yaxes(autorange="reversed")
fig.update_layout(height=650)
save_plot(fig, "hcm_fomo_vs_ttfl.html")

# -----------------------------
# 13F. Segment count
# -----------------------------
seg_city = (
    city_demand["demand_segment"]
    .value_counts()
    .reset_index()
)
seg_city.columns = ["demand_segment", "groups"]

fig = px.bar(
    seg_city,
    x="demand_segment",
    y="groups",
    title="City Group Count by Demand Segment",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
save_plot(fig, "city_segment_count.html")

seg_hcm = (
    hcm_demand["demand_segment"]
    .value_counts()
    .reset_index()
)
seg_hcm.columns = ["demand_segment", "groups"]

fig = px.bar(
    seg_hcm,
    x="demand_segment",
    y="groups",
    title="HCM Group Count by Demand Segment",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
save_plot(fig, "hcm_segment_count.html")

# ============================================================
# 14. OUTPUT FILES
# ============================================================

print("\nDONE. Output files:")
for f in sorted(os.listdir(CACHE_DIR)):
    print("-", os.path.join(CACHE_DIR, f))

Available Kaggle input folders:
/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/fact_listing_snapshot | sample: ['datathon_fact_listing_snapshot-000000000052.parquet', 'datathon_fact_listing_snapshot-000000000036.parquet']
/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/fact_post_contact_interactions | sample: ['datathon_fact_user_ad_interactions-000000000092.parquet', 'datathon_fact_user_ad_interactions-000000000099.parquet']
/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/dim_listing | sample: ['datathon_dim_listing-000000000012.parquet', 'datathon_dim_listing-000000000023.parquet']
/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/test | sample: ['test_users.parquet']
/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events | sample: ['datathon_fact_user_events-000000000246.parquet', 'datathon_fact_user_events-000000000185.parquet']

[1/7] Building item_map from dim_listing...
dim_listing batches: 10
dim_listing batches: 20
dim_listing b

,item_id,city_name,district_name,category,category_name,ad_type,posted_date
0,bc8a95ba9c8c41b86fc446e58af07aacf1e98cb0a8f632...,Tp Hồ Chí Minh,Quận 3,1010,1010 - Phòng trọ / căn hộ thuê,let,2025-03-04
1,2e58d907f4986a98aa31700ae56883bbc85748932d58af...,Tp Hồ Chí Minh,Quận 8,1010,1010 - Phòng trọ / căn hộ thuê,let,2025-05-04
2,3d186c9e829ba17729a239816bcfe24c9e23bdd119f62c...,Tp Hồ Chí Minh,Quận 8,1010,1010 - Phòng trọ / căn hộ thuê,let,2025-04-17
3,e9f549a8c6b0ed5c823df7de5c0344d7b0b1777a65af61...,Tp Hồ Chí Minh,Quận 3,1010,1010 - Phòng trọ / căn hộ thuê,let,2025-07-03
4,6c30deece657bdce3cd0ab779e3dc0954ca775cd275379...,Tp Hồ Chí Minh,Quận Gò Vấp,1010,1010 - Phòng trọ / căn hộ thuê,let,2025-11-26


City group keys: ['city_name', 'category_name', 'ad_type']
HCM group keys: ['district_name', 'category_name', 'ad_type']

[2/7] Building supply tables...
city_supply: (508, 4)
hcm_supply: (200, 4)


,city_name,category_name,ad_type,listing_count
407,Tp Hồ Chí Minh,1020 - Căn hộ / chung cư,sell,733140
404,Tp Hồ Chí Minh,1010 - Phòng trọ / căn hộ thuê,let,373553
412,Tp Hồ Chí Minh,1050 - Dự án / căn hộ mới,let,325650
406,Tp Hồ Chí Minh,1020 - Căn hộ / chung cư,let,321968
408,Tp Hồ Chí Minh,1030 - Nhà ở,let,175926
133,Hà Nội,1020 - Căn hộ / chung cư,sell,160037
411,Tp Hồ Chí Minh,1040 - Đất / BĐS thương mại,sell,99867
405,Tp Hồ Chí Minh,1010 - Phòng trọ / căn hộ thuê,sell,79612
467,Đà Nẵng,1020 - Căn hộ / chung cư,sell,57231
471,Đà Nẵng,1040 - Đất / BĐS thương mại,sell,48328


,district_name,category_name,ad_type,listing_count
147,Quận Bình Tân,1020 - Căn hộ / chung cư,sell,90354
156,Quận Gò Vấp,1020 - Căn hộ / chung cư,sell,84777
75,Quận 12,1020 - Căn hộ / chung cư,sell,71540
193,Thành phố Thủ Đức,1020 - Căn hộ / chung cư,sell,70658
183,Quận Tân Phú,1020 - Căn hộ / chung cư,sell,59114
161,Quận Gò Vấp,1050 - Dự án / căn hộ mới,let,51298
135,Quận Bình Thạnh,1010 - Phòng trọ / căn hộ thuê,let,46815
174,Quận Tân Bình,1020 - Căn hộ / chung cư,sell,46272
138,Quận Bình Thạnh,1020 - Căn hộ / chung cư,sell,45568
120,Quận 7,1020 - Căn hộ / chung cư,sell,44566


...

[SAVED] /kaggle/working/demand_v3_fomo_ttfl_traffic/top_city_demand_score.html


[SAVED] /kaggle/working/demand_v3_fomo_ttfl_traffic/top_hcm_demand_score.html


[SAVED] /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_fomo_vs_ttfl.html


[SAVED] /kaggle/working/demand_v3_fomo_ttfl_traffic/city_segment_count.html


[SAVED] /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_segment_count.html

DONE. Output files:
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_demand_v3_fomo_ttfl_traffic.csv
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_fomo.parquet
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_groups_too_small.csv
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_segment_count.html
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_supply.parquet
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_supply_vs_demand_score.html
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_traffic.parquet
- /kaggle/working/demand_v3_fomo_ttfl_traffic/city_ttfl.parquet
- /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_district_demand_v3_fomo_ttfl_traffic.csv
- /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_fomo.parquet
- /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_fomo_vs_ttfl.html
- /kaggle/working/demand_v3_fomo_ttfl_traffic/hcm_groups_too_small.csv
- /kaggle/working/demand_v3_fomo_ttfl_traffic/